# Pipeline 2 — Reintegration Drivers (Explanatory)

## 1) Problem Framing

**Business question:** Which factors most strongly *explain* reintegration completion (not only predict it), and in what direction?

- **Type:** Explanatory / inference-oriented (interpretable coefficients).
- **Target:** `reintegration_complete` — same definition as Pipeline 1 (`reintegration_status == 'Completed'`).
- **Primary metrics:** Adjusted R² (OLS), coefficient significance (p-values), pseudo-R² (logistic comparison).
- **Operational use:** Publish **one org-level insight** row to `ml_predictions` with top drivers for impact/donor messaging.

### Prediction vs explanation (textbook framing)

Pipeline 1 prioritizes **predictive accuracy** (ROC-AUC, held-out discrimination). Pipeline 2 prioritizes **interpretable associations** (linear coefficients on a defensible feature set, multicollinearity control). The same resident-month features can support both goals, but feature selection and evaluation criteria differ: here we avoid RFECV-for-prediction as the final selector and emphasize VIF and domain reasoning.


In [1]:
# 2) Data Acquisition and Preparation — import shared features (do not rewrite)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor

PROJECT_ROOT = Path.cwd().parents[1]
ML_DIR = PROJECT_ROOT / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.append(str(ML_DIR))

from config import (  # noqa: E402
    MODEL_REINTEGRATION_DRIVERS,
    MODEL_RUNS_REINTEGRATION_DRIVERS,
)
from reintegration_readiness.etl import build_training_frame  # noqa: E402
from reintegration_drivers.artifacts import (  # noqa: E402
    save_metadata,
    save_metrics,
    save_model_bundle,
)

RANDOM_STATE = 42

In [2]:
train_df = build_training_frame()

assert "reintegration_complete" in train_df.columns
assert train_df["resident_id"].is_unique

y = train_df["reintegration_complete"].astype(int)
X = train_df.drop(columns=["reintegration_complete", "resident_id"], errors="ignore")

print("Rows:", len(train_df))
print("Feature count:", X.shape[1])
print("Class distribution:\n", y.value_counts())


Rows: 60
Feature count: 34
Class distribution:
 reintegration_complete
0    41
1    19
Name: count, dtype: int64


## 3) Exploration

Pipeline 1 already documented correlations and univariate structure. Highlights to keep in mind:

- `visits_per_month` is among the stronger linear associations with completion.
- Case category (one-hot) shifts baseline completion rates materially.
- With **n≈60**, coefficients are unstable; treat inference as **exploratory** and prioritize direction/significance over precise magnitudes.

Below: quick correlation with the target (reference only — not repeating every Pipeline 1 chart).


In [3]:
corr = train_df.drop(columns=["resident_id"], errors="ignore").corr(numeric_only=True)
print(corr["reintegration_complete"].sort_values(ascending=False).head(15))


reintegration_complete       1.000000
trauma_severity_score        0.293028
case_category_Surrendered    0.251643
visits_per_month             0.247396
total_visits                 0.196657
age_at_admission             0.169407
medical_checkups             0.157580
post_placement_visits        0.154898
checkup_compliance           0.117901
psych_checkups               0.088594
current_risk_num             0.077448
initial_risk_num             0.075298
avg_attendance               0.065859
favorable_rate               0.049880
length_of_stay_months        0.048761
Name: reintegration_complete, dtype: float64


## 4) Feature Selection — interpretability & multicollinearity (Ch. 16 style)

Steps (no RFECV as final selector):

1. Drop near-zero variance.
2. Drop pairwise redundant features (|r| > 0.95) on the training split.
3. Iterative **VIF** pruning (drop the highest VIF feature while any VIF > 10) to stabilize coefficients.
4. Optional domain drops: document any removals that are mediators/proxies.

Target **~8–12** interpretable predictors where possible.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Near-zero variance
variances = X_train.var(numeric_only=True)
keep_var = variances[variances > 1e-12].index.tolist()
X_train_v = X_train[keep_var].copy()
X_test_v = X_test[keep_var].copy()

# High correlation pruning (training only)
corr_mat = X_train_v.corr(numeric_only=True).abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
X_train_f = X_train_v.drop(columns=drop_corr, errors="ignore")
X_test_f = X_test_v.drop(columns=drop_corr, errors="ignore")
print("After variance + correlation pruning:", X_train_f.shape[1], "features")


def compute_vif(frame: pd.DataFrame) -> pd.DataFrame:
    vals = frame.values.astype(float)
    vifs = []
    for i in range(vals.shape[1]):
        vifs.append(variance_inflation_factor(vals, i))
    return pd.DataFrame({"feature": frame.columns, "VIF": vifs})


X_vif = X_train_f.copy()
while True:
    vif_df = compute_vif(X_vif)
    vmax = vif_df["VIF"].replace([np.inf, -np.inf], np.nan).max()
    if np.isnan(vmax) or vmax <= 10:
        break
    worst = vif_df.sort_values("VIF", ascending=False).iloc[0]["feature"]
    X_vif = X_vif.drop(columns=[worst])
    print("Dropped (high VIF):", worst, "max VIF was", vmax)

X_sel = X_vif.copy()
X_test_sel = X_test_f[X_sel.columns]
print("Final feature count:", X_sel.shape[1])
print(compute_vif(X_sel))


After variance + correlation pruning: 29 features
Dropped (high VIF): initial_risk_num max VIF was 557.9573590686064
Dropped (high VIF): case_category_Surrendered max VIF was 557.9573590686064
Dropped (high VIF): avg_attendance max VIF was 477.4799499792898
Dropped (high VIF): avg_health max VIF was 138.14328205998137
Dropped (high VIF): checkup_compliance max VIF was 58.80489732238087
Dropped (high VIF): avg_progress max VIF was 48.173121134665074
Dropped (high VIF): total_visits max VIF was 33.509740280553885
Dropped (high VIF): age_at_admission max VIF was 22.237142651681207
Dropped (high VIF): family_coop_rate max VIF was 19.46498610897635
Dropped (high VIF): medical_checkups max VIF was 14.468434884331282
Dropped (high VIF): visits_per_month max VIF was 13.070444337343455
Dropped (high VIF): length_of_stay_days max VIF was 11.934061398948971
Final feature count: 17
                       feature       VIF
0             current_risk_num  7.663096
1               risk_reduction  2.0

C:\Users\justi\AppData\Roaming\Python\Python314\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


## 5–6) Modeling — OLS (primary), logistic (comparison), tree (sanity check)

- Fit **OLS** on **standardized** training features (coefficients are comparable "per SD").
- Fit **Logit** on the same design matrix for direction/significance comparison on the binary target.
- Fit a shallow **Decision Tree** only as a non-primary sanity check (not the deliverable).


In [5]:
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_sel)
X_test_std = scaler.transform(X_test_sel)

X_train_sm = sm.add_constant(pd.DataFrame(X_train_std, columns=X_sel.columns, index=X_sel.index))
X_test_sm = sm.add_constant(pd.DataFrame(X_test_std, columns=X_sel.columns, index=X_test_sel.index))

ols = sm.OLS(y_train, X_train_sm).fit()
print(ols.summary())

logit = None
try:
    logit = sm.Logit(y_train, X_train_sm).fit(disp=0)
    print("\nLogit pseudo R2 (McFadden):", getattr(logit, "prsquared", np.nan))
    print(logit.summary())
except Exception as e:
    print("\nLogit fit skipped due to model singularity/instability:", e)

tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=4, random_state=RANDOM_STATE)
tree.fit(X_train_std, y_train)
print("Tree train acc:", tree.score(X_train_std, y_train))
print("Tree test acc:", tree.score(X_test_std, y_test))


                              OLS Regression Results                              
Dep. Variable:     reintegration_complete   R-squared:                       0.604
Model:                                OLS   Adj. R-squared:                  0.379
Method:                     Least Squares   F-statistic:                     2.690
Date:                    Tue, 07 Apr 2026   Prob (F-statistic):            0.00866
Time:                            11:39:35   Log-Likelihood:                -8.9778
No. Observations:                      48   AIC:                             53.96
Df Residuals:                          30   BIC:                             87.64
Df Model:                              17                                         
Covariance Type:                nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------

C:\Users\justi\AppData\Roaming\Python\Python314\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
C:\Users\justi\AppData\Roaming\Python\Python314\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


## Evaluation — explanatory diagnostics

- **Adjusted R²** (OLS) and **pseudo-R²** (logistic).
- **Coefficient table** with p-values and CIs (from OLS summary).
- **VIF** on the final standardized training matrix (flag VIF > 5).
- **Residual checks** (OLS): rough normality of residuals + heteroskedasticity note.


In [6]:
print("Adjusted R2 (train):", ols.rsquared_adj)

final_vif = compute_vif(pd.DataFrame(X_train_std, columns=X_sel.columns))
print("\nFinal VIF:\n", final_vif)
print("\nFlags VIF > 5:", final_vif[final_vif["VIF"] > 5]["feature"].tolist())

resid = ols.resid
print("Residual Shapiro p (normality):", stats.shapiro(resid)[1])

try:
    from statsmodels.stats.diagnostic import het_breuschpagan
    bp = het_breuschpagan(resid, X_train_sm)
    print("Breusch-Pagan p-value:", bp[1])
except Exception as e:
    print("het_breuschpagan skipped:", e)


Adjusted R2 (train): 0.37936322112895526

Final VIF:
                        feature       VIF
0             current_risk_num  2.058251
1               risk_reduction  1.554652
2        trauma_severity_score  1.845365
3   family_vulnerability_score  1.272366
4               psych_checkups  1.425504
5                 health_trend  1.695954
6            courses_completed  1.445142
7               total_sessions  5.845774
8        positive_session_rate  5.908962
9               favorable_rate  1.535502
10         safety_concern_rate  2.078860
11       post_placement_visits  2.297233
12   reintegration_assessments  1.996311
13  intervention_achieved_rate  1.120769
14     case_category_Abandoned  1.712175
15     case_category_Foundling  1.891789
16     case_category_Neglected  1.889137

Flags VIF > 5: ['total_sessions', 'positive_session_rate']
Residual Shapiro p (normality): 0.7306144853059552
Breusch-Pagan p-value: 0.11223290009493138


## 7) Causal and Relationship Analysis

These models estimate **associations** under linear/logistic functional forms. They are **not** causal impacts without stronger design (randomization, instruments, or rich confounder control).

- **Confounding:** intensity of visits/counseling may be higher for girls who are already progressing (reverse causality is plausible).
- **Defensible claims:** directionally sensible coefficients that survive multicollinearity scrutiny are **hypothesis-generating** for leadership and donors.
- **Stronger causal claims** would require prospective randomization or quasi-experiments — say that explicitly in reporting.


## 8) Deployment Notes

- Train artifacts are saved to `models/reintegration-drivers.sav` plus JSON metadata/metrics.
- Nightly infer (`pipelines/reintegration_drivers_infer.py`) writes **one** `org_insight` row (not per resident).
- Re-run the notebook after material data refreshes; keep `training_date` in metadata for versioning.


In [7]:
# Ch. 17 — Save artifacts


def coef_table(fit) -> list[dict]:
    rows = []
    conf = fit.conf_int()
    for name in fit.params.index:
        if name == "const":
            continue
        lo = float(conf.loc[name].iloc[0])
        hi = float(conf.loc[name].iloc[1])
        rows.append(
            {
                "feature": name,
                "coefficient": float(fit.params[name]),
                "std_err": float(fit.bse[name]),
                "p_value": float(fit.pvalues[name]),
                "ci_lower": lo,
                "ci_upper": hi,
            }
        )
    return rows


ols_rows = coef_table(ols)
logit_rows = coef_table(logit) if logit is not None else None

save_model_bundle(ols, scaler, list(X_sel.columns), logit)

meta = save_metadata(
    model_type="OLS (statsmodels) + Logit comparison",
    feature_list=list(X_sel.columns),
    train_rows=len(X_train),
    test_rows=len(X_test),
    total_rows=len(train_df),
)

metrics = save_metrics(
    adjusted_r2=float(ols.rsquared_adj),
    pseudo_r2=float(logit.prsquared) if (logit is not None and hasattr(logit, "prsquared")) else None,
    n_observations=int(len(y_train)),
    ols_coefficients=ols_rows,
    logit_coefficients=logit_rows,
)

print("Saved:")
print(" -", MODEL_REINTEGRATION_DRIVERS)
print(" -", MODEL_RUNS_REINTEGRATION_DRIVERS)
print("metadata keys:", meta.keys())
print("metrics keys:", metrics.keys())


Saved:
 - C:\Users\justi\Desktop\BYU\INTEX\intex2-1\models\reintegration-drivers\model.sav
 - C:\Users\justi\Desktop\BYU\INTEX\intex2-1\models\reintegration-drivers\model.json
metadata keys: dict_keys(['model_name', 'model_version', 'trained_at_utc', 'features', 'num_training_rows', 'num_test_rows', 'training_date', 'model_type', 'feature_list', 'train_rows', 'test_rows', 'total_rows'])
metrics keys: dict_keys(['model_name', 'model_version', 'trained_at_utc', 'features', 'num_training_rows', 'num_test_rows', 'training_date', 'model_type', 'feature_list', 'train_rows', 'test_rows', 'total_rows', 'accuracy', 'f1', 'roc_auc', 'classification_report', 'adjusted_r2', 'pseudo_r2', 'n_observations', 'ols_coefficients'])
